# 카카오 로컬(Local) API 예시

주소 검색 · 지오코딩 · 역지오코딩 · 키워드 검색을 순서대로 실행해 봅니다.

**준비**
1. [카카오 개발자 콘솔](https://developers.kakao.com)에서 앱 생성 → **REST API 키** 발급
2. 프로젝트 루트에 `.env` 파일을 만들고 `KAKAO_REST_API_KEY=발급받은키` 입력

**주의**: 카카오 API는 `x = 경도(longitude)`, `y = 위도(latitude)` 순서입니다.

## 0. 준비

In [ ]:
# 최초 1회만 실행
# !pip install requests python-dotenv pandas

In [ ]:
import os
import time

import requests
from dotenv import load_dotenv

load_dotenv()                              # .env 에서 키를 읽는다
API_KEY = os.getenv("KAKAO_REST_API_KEY")

# .env 를 안 쓴다면 아래처럼 입력 (노트북에 키를 직접 적어 저장하지 말 것)
# import getpass
# API_KEY = getpass.getpass("REST API KEY: ")

BASE = "https://dapi.kakao.com/v2/local"
HEADERS = {"Authorization": f"KakaoAK {API_KEY}"}

print("키 설정됨:", bool(API_KEY))

## 1. 주소 검색

`/search/address.json` — 주소 문자열로 검색합니다. 도로명·지번 어느 쪽이든 됩니다.

In [ ]:
def search_address(query, size=10, page=1):
    """주소를 검색해 documents 리스트를 반환한다."""
    res = requests.get(
        f"{BASE}/search/address.json",
        headers=HEADERS,
        params={"query": query, "size": size, "page": page},
    )
    return res.json()["documents"]


docs = search_address("판교역로 235")
print(f"{len(docs)}건")

for d in docs:
    road = d.get("road_address") or {}
    print(f"{d['address_name']:35s} | {road.get('address_name', '-'):35s} | {d['x']}, {d['y']}")

In [ ]:
# 응답 구조가 궁금하면 원본을 그대로 확인
docs[0]

## 2. 지오코딩 (주소 → 좌표)

주소 검색 결과에서 필요한 값만 뽑아 정리합니다.

In [ ]:
def geocode(address):
    """주소를 좌표로 변환한다. 결과가 없으면 None."""
    docs = search_address(address, size=1)
    if not docs:
        return None

    d = docs[0]
    road = d.get("road_address") or {}
    jibun = d.get("address") or {}
    return {
        "input": address,
        "lng": float(d["x"]),
        "lat": float(d["y"]),
        "road_address": road.get("address_name", ""),
        "jibun_address": jibun.get("address_name", ""),
        "zone_no": road.get("zone_no", ""),          # 우편번호
        "building": road.get("building_name", ""),
        "sido": jibun.get("region_1depth_name", ""),
        "sigungu": jibun.get("region_2depth_name", ""),
        "dong": jibun.get("region_3depth_name", ""),
    }


center = geocode("경기도 성남시 분당구 판교역로 235")
center

### 2-1. 여러 주소를 한 번에

In [ ]:
addresses = [
    "서울 중구 세종대로 110",          # 서울시청
    "서울 용산구 남산공원길 105",       # N서울타워
    "부산 해운대구 해운대해변로 264",
    "제주 제주시 첨단로 242",
    "없는주소 12345",                  # 실패 케이스
]

rows = []
for addr in addresses:
    rows.append(geocode(addr))
    time.sleep(0.05)                   # 호출 간격(쿼터 보호)

for addr, r in zip(addresses, rows):
    if r:
        print(f"{addr:30s} -> ({r['lat']:.6f}, {r['lng']:.6f})  {r['building']}")
    else:
        print(f"{addr:30s} -> 결과 없음")

## 3. 역지오코딩 (좌표 → 주소)

- `/geo/coord2address.json` — 도로명 / 지번 주소
- `/geo/coord2regioncode.json` — 행정동 · 법정동과 행정코드

In [ ]:
def reverse_geocode(lng, lat):
    """좌표를 주소로 변환한다."""
    res = requests.get(
        f"{BASE}/geo/coord2address.json",
        headers=HEADERS,
        params={"x": lng, "y": lat},
    )
    docs = res.json()["documents"]
    if not docs:
        return None

    road = docs[0].get("road_address") or {}
    jibun = docs[0].get("address") or {}
    return {
        "road_address": road.get("address_name", ""),
        "jibun_address": jibun.get("address_name", ""),
        "zone_no": road.get("zone_no", ""),
        "building": road.get("building_name", ""),
    }


def coord_to_region(lng, lat):
    """좌표의 행정동/법정동 정보를 반환한다."""
    res = requests.get(
        f"{BASE}/geo/coord2regioncode.json",
        headers=HEADERS,
        params={"x": lng, "y": lat},
    )
    return res.json()["documents"]


print(reverse_geocode(center["lng"], center["lat"]))
print()

for r in coord_to_region(center["lng"], center["lat"]):
    # region_type: B=법정동, H=행정동
    print(f"[{r['region_type']}] {r['address_name']}  code={r['code']}")

## 4. 키워드 검색

`/search/keyword.json` — 장소명·업종명 등으로 검색합니다.
중심 좌표(`x`, `y`)와 `radius`(m, 최대 20000)를 주면 반경 검색이 됩니다.

In [ ]:
def search_keyword(query, lng=None, lat=None, radius=None,
                   page=1, size=15, sort=None, category_group_code=None):
    """키워드로 장소를 검색한다. 원본 응답(meta + documents)을 반환.

    sort: "accuracy"(기본, 정확도순) | "distance"(거리순, 좌표 필요)
    """
    params = {
        "query": query,
        "x": lng,
        "y": lat,
        "radius": radius,
        "page": page,
        "size": size,
        "sort": sort,
        "category_group_code": category_group_code,
    }
    params = {k: v for k, v in params.items() if v is not None}

    res = requests.get(f"{BASE}/search/keyword.json", headers=HEADERS, params=params)
    return res.json()


data = search_keyword(
    "카페",
    lng=center["lng"],
    lat=center["lat"],
    radius=1000,
    sort="distance",
    size=10,
)

print("전체:", data["meta"]["total_count"], "건 / 노출 가능:", data["meta"]["pageable_count"], "건")
print()

for d in data["documents"]:
    print(f"{d['distance']:>5s}m  {d['place_name']:25s} {d['road_address_name']}")

### 4-1. 여러 페이지 모으기

키워드/카테고리 검색은 **한 조건당 최대 45건**까지만 노출됩니다.
그보다 많이 모으려면 반경을 격자로 쪼개거나 키워드를 나눠야 합니다.

In [ ]:
def search_keyword_all(query, max_results=45, **kwargs):
    """is_end 가 될 때까지 페이지를 넘기며 모은다 (최대 45건)."""
    kwargs.pop("page", None)
    kwargs.pop("size", None)

    results = []
    for page in range(1, 46):
        data = search_keyword(query, page=page, size=15, **kwargs)
        results.extend(data["documents"])

        if data["meta"]["is_end"] or len(results) >= max_results:
            break
        time.sleep(0.05)

    return results[:max_results]


places = search_keyword_all(
    "카페", lng=center["lng"], lat=center["lat"], radius=1000, sort="distance"
)
print(len(places), "건 수집")

## 5. 카테고리 검색

`/search/category.json` — 업종 코드로 주변을 훑을 때 키워드보다 정확합니다.

In [ ]:
CATEGORY_GROUP = {
    "대형마트": "MT1", "편의점": "CS2", "어린이집·유치원": "PS3",
    "학교": "SC4", "학원": "AC5", "주차장": "PK6",
    "주유소·충전소": "OL7", "지하철역": "SW8", "은행": "BK9",
    "문화시설": "CT1", "중개업소": "AG2", "공공기관": "PO3",
    "관광명소": "AT4", "숙박": "AD5", "음식점": "FD6",
    "카페": "CE7", "병원": "HP8", "약국": "PM9",
}


def search_category(code, lng, lat, radius=1000, page=1, size=15, sort="distance"):
    """카테고리 코드로 주변 장소를 검색한다."""
    res = requests.get(
        f"{BASE}/search/category.json",
        headers=HEADERS,
        params={
            "category_group_code": code,
            "x": lng,
            "y": lat,
            "radius": radius,
            "page": page,
            "size": size,
            "sort": sort,
        },
    )
    return res.json()["documents"]


for name in ["지하철역", "학교", "대형마트"]:
    found = search_category(
        CATEGORY_GROUP[name], center["lng"], center["lat"], radius=1500
    )
    print(f"--- {name} ({len(found)}건) ---")
    for d in found[:5]:
        print(f"  {d['distance']:>5s}m  {d['place_name']}")
    print()

## 6. 결과를 표로 정리

수집한 결과를 `pandas.DataFrame` 으로 옮기면 필터링·집계가 편합니다.

In [ ]:
import pandas as pd

df = pd.DataFrame(places)[
    ["place_name", "category_name", "road_address_name", "phone", "distance", "x", "y"]
]
df = df.rename(columns={"x": "lng", "y": "lat"})
df["distance"] = df["distance"].astype(int)
df[["lng", "lat"]] = df[["lng", "lat"]].astype(float)

df.head(10)

In [ ]:
# 예: 500m 이내만, 카테고리 소분류별 개수
near = df[df["distance"] <= 500]
print(f"500m 이내 {len(near)}건")

near["category_name"].str.split(" > ").str[-1].value_counts()

In [ ]:
# CSV 로 저장
df.to_csv("kakao_places.csv", index=False, encoding="utf-8-sig")
print("저장 완료: kakao_places.csv")

---
## 참고

| 항목 | 내용 |
| --- | --- |
| 좌표 순서 | `x` = 경도(lng), `y` = 위도(lat) |
| 결과 상한 | 키워드·카테고리 검색은 조건당 최대 **45건** |
| `size` | 주소 검색 1~30, 키워드·카테고리 1~15 |
| `radius` | 0~20000 (m) |
| 쿼터 초과 | HTTP `429` 응답 — 호출 간격을 두거나 결과를 캐싱 |
| 인증 실패 | HTTP `401` — REST API 키인지, 헤더가 `KakaoAK <키>` 형식인지 확인 |

[카카오 로컬 API 문서](https://developers.kakao.com/docs/latest/ko/local/dev-guide)